## 다중 선형회귀
> 광고비, 가격, 시즌(계절), 유통채널(자사몰, 오픈마켓, 라이브커머스)에 따른 제품 매출액 예측

In [1]:
#1. 데이터 불러오기
import pandas as pd
df= pd.read_csv('/content/drive/MyDrive/MBCA/Machine Learning/04 Multiple Linear Regression/marketing_sales_data.csv')
df

# 컬룸정보


,ad_spend,price,season,channel,sales
0,5000000,32000,winter,D2C,21000000
1,5500000,32000,winter,open_market,23000000
2,5300000,32500,winter,live_commerce,22500000
3,6000000,33000,winter,live_commerce,25500000
4,6000000,31000,spring,D2C,27000000
5,6500000,30000,spring,live_commerce,30000000
6,7000000,30000,spring,open_market,32000000
7,5800000,31000,spring,D2C,26000000
8,6200000,30000,spring,live_commerce,29000000
9,6800000,29000,spring,open_market,31000000


In [2]:
df.shape

(35, 5)

In [3]:
df.info()
#컬룸 정보
# ad_spend 광고비 수치형데이터
# price    가격   수치형데이터
# season   계절   범주형데이터
# channel  유통채널 범주형데이터

# sales    매출액 수치형

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ad_spend  35 non-null     int64 
 1   price     35 non-null     int64 
 2   season    35 non-null     object
 3   channel   35 non-null     object
 4   sales     35 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 1.5+ KB


In [4]:
# 선형회귀는 직선의 방정식을 사용하기에.. 수치형데이터만 요인으로 가중치를 계산할 수 있음.
# 그래서. 범주형 데이터를 숫자로 변환하여 학습을 시킴 - 전처리

# 보통 남자/여자 , 합격/불합격 같은 2개의 범주형 값은 1/0 으로 변환하여 독립변수로 활용함
# 카테고리의 값이 3개 이상이면??
# 예) 시즌정보 [spring, summer, fall, winter]
# 이 4개의 값을 숫자로 다르게 표현 하려면.. 숫자 4개를 사용
# spring=0, summer=1, fall=2, winter=3

# 이러면 될까?? 선형회귀는 가중치*x
# winter가 같은 가중치를 가지고 있더라도 거 크데 영향력을 주는 것처럼 계산됨.

# 이럴때 사용하는 가장 기본적인 방법. One-hot encoding
# 시즌              -> 파생변수     spring  |   summer   |    fall    |      winter
# spring            ->                 1          0            0               0
# summer            ->                 0          1            0               0
# fall              ->                 0          0            1               0
# winter            ->                 0          0            0               1

# One-hot encoding을 해주는 기능이 데이터프레임에 이미 존재함
encoded_df= pd.get_dummies(df) # 데이터프레임안에 있는 모든 범주형 컬룸들을 인코딩함.
encoded_df.head(3) #버전업이 되어 0/1 대신에 True/False로 변환함. bool 타입. (메모리 효율). LinearRegression이 알아서 숫자로 바꿔서 계산함.

,ad_spend,price,sales,season_fall,season_spring,season_summer,season_winter,channel_D2C,channel_live_commerce,channel_open_market
0,5000000,32000,21000000,False,False,False,True,True,False,False
1,5500000,32000,23000000,False,False,False,True,False,False,True
2,5300000,32500,22500000,False,False,False,True,False,True,False


In [5]:
# True/False 대신 숫자 1/0 으로 만들고 싶다면..
encoded_df= pd.get_dummies(df, dtype=int)
encoded_df.head(3)

,ad_spend,price,sales,season_fall,season_spring,season_summer,season_winter,channel_D2C,channel_live_commerce,channel_open_market
0,5000000,32000,21000000,0,0,0,1,1,0,0
1,5500000,32000,23000000,0,0,0,1,0,0,1
2,5300000,32500,22500000,0,0,0,1,0,1,0


In [6]:
# 특정 컬룸의 범주형 데이터만 인코딩 하고 싶다면.
encoded_df=pd.get_dummies(df, dtype=int, columns=['channel'])
encoded_df.tail(3)

,ad_spend,price,season,sales,channel_D2C,channel_live_commerce,channel_open_market
32,5700000,33000,winter,22000000,0,0,1
33,5900000,33000,winter,22500000,0,0,1
34,6200000,33000,winter,24500000,1,0,0


In [8]:
# 범주형의 원핫인코딩 변수들 중 마지막 변수는 없어도 앞의 변수들의 값을 기반으로 구별이 가능
# 다중공선성 문제를 해결하기 위해 첫번째 변수는 제거

# 실습의 편의를 위해.. season 컬룸은 사용하지 않고.. channel 만 사용.
encoded_df= pd.get_dummies(df, dtype=int, columns=['channel'], drop_first=True)
encoded_df.drop('season', axis=1, inplace=True)
encoded_df.head(3)

,ad_spend,price,sales,channel_live_commerce,channel_open_market
0,5000000,32000,21000000,0,0
1,5500000,32000,23000000,0,1
2,5300000,32500,22500000,1,0


In [10]:
#3. 독립변수와 종속변수를 분리
X= encoded_df.drop('sales', axis=1) # 종속변수만 제거..
y= encoded_df['sales']

In [11]:
#4. 훈련/트세트 데이터셋 분리
from sklearn.model_selection import train_test_split
train_feature, test_feature, train_target, test_target= train_test_split(X,y, test_size=0.2, random_state=42)

print(train_feature.shape, train_target.shape)
print(test_feature.shape, test_target.shape)

(28, 4) (28,)
(7, 4) (7,)


In [13]:
#5. 모델 생성 및 학습
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(train_feature, train_target)# 훈련용 독립변수, 훈련용 종속변수

LinearRegression()

In [14]:
y_pred=model.predict(test_feature)
y_pred

array([30834546.75993443, 37296955.55230834, 27672691.79097781,
       43829709.82278857, 41841080.60864069, 30414492.90941254,
       38865530.91593432])

In [15]:
#7. 예측모델의 성능 평가
from sklearn.metrics import mean_squared_error, r2_score
print('R2 score:', r2_score(test_target, y_pred))
print('MSE', mean_squared_error(test_target, y_pred))

R2 score: 0.9908806285896867
MSE 304754503763.0199


In [16]:
#8. 회귀계수(가중치, 절편) 확인
print('독립변수들의 가중치들:', model.coef_)
print('독립변수들의 절편:', model.intercept_)

독립변수들의 가중치들: [ 4.20053851e+00 -1.14852151e+03  1.22108536e+05 -8.04934962e+05]
독립변수들의 절편: 36999771.3905766


In [18]:
#[최종] 새로운 데이터를 주고 매출액 예측해보기
new_data= pd.DataFrame({
    'ad_spend': [7500000], #750만원
    'price':[30000],
    'season':['spring'],
    'channel':['open_market']
})

#이 데이터도 범주형이 포함되어 있으니.. 모델이 학습했던 독립변수 구조로 만들어야 함.
new_data_encoded= pd.get_dummies(new_data, dtype=int, columns=['channel'], drop_first=True)
new_data_encoded.drop('season', axis=1)

# 컬룸들을 조정하다보면 훈련용으로 만들었던 독립변수들의 컬룸순서와 다를 수 있음.
# 훈련용 데이터셋과 구조를 맞추기
new_data_encoded= new_data_encoded.reindex(columns=train_feature.columns, fill_value=0)

# 예측 요청.
prediction_result=model.predict(new_data_encoded)
print('예상 매출액:', prediction_result[0])

예상 매출액: 34048164.78659479
